<a href="https://colab.research.google.com/github/hmnaz213/native/blob/main/alibaba_shipping_forwarder_order_management.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Load Google Sheet Data

In [22]:
import pandas as pd
import gspread

In [23]:
import pandas as pd
import gspread
from google.colab import auth
from google.auth import default
from datetime import datetime

try:
    # Authenticate user with Google Colab
    auth.authenticate_user()

    # Get default credentials
    creds, _ = default()

    # Authorize gspread with the obtained credentials
    gc = gspread.authorize(creds)
    print("Authenticated successfully with Google Colab.")

    spreadsheet_url = 'https://docs.google.com/spreadsheets/d/13rC6ZVcKKeHAeYoJPi86s-6VO6g0k4cRkGWzBeGwvlo/edit?gid=463685144#gid=463685144'

    # The GID is provided in the URL and is 463685144
    worksheet_gid = 463685144

    # Open the spreadsheet by URL
    sh = gc.open_by_url(spreadsheet_url)

    # Select the worksheet by its gid
    worksheet = sh.get_worksheet_by_id(worksheet_gid)

    # Get all values from the worksheet as a list of lists
    data = worksheet.get_all_values()

    # Convert to a pandas DataFrame. The first row is assumed to be the header.
    if data:
        # Clean header: remove any empty strings that might be trailing
        header = [h for h in data[0] if h]
        num_columns = len(header)

        # Helper function to check if a string can be parsed as a date
        def is_date(s, format_str='%d/%m/%Y'):
            if not isinstance(s, str):
                return False
            try:
                datetime.strptime(s, format_str)
                return True
            except (ValueError, TypeError):
                return False

        processed_data = []
        for row_idx, row in enumerate(data[1:]):
            current_row = row[:] # Make a copy to modify

            # Heuristic 1: Check for leading shift (e.g., 'ssot/alibaba_order_creation' in email_date)
            # This applies if the first element is NOT a date but the second is.
            if len(current_row) > 1 and not is_date(current_row[0]) and is_date(current_row[1]):
                current_row = current_row[1:]

            # Heuristic 2: Check for '0' in the 'ingested_at' position, shifting valid data right
            # 'ingested_at' is typically the 9th column (index 8) after a clean header of 10.
            # If current_row[8] is '0' and current_row[9] is a valid date (expected for ingested_at)
            # then remove the '0' at index 8.
            if (len(current_row) > 8 and current_row[8] == '0' and
                len(current_row) > 9 and is_date(current_row[9])):
                current_row.pop(8) # Remove the '0'

            # Final step: Normalize row length to match the clean header, padding or truncating
            if len(current_row) < num_columns:
                current_row = current_row + [None] * (num_columns - len(current_row))
            elif len(current_row) > num_columns:
                current_row = current_row[:num_columns]

            processed_data.append(current_row)

        df = pd.DataFrame(processed_data, columns=header)
        print("Google Sheet loaded successfully into DataFrame 'df'.")
        print("First 5 rows of the DataFrame (after improved alignment):")
        display(df.head())
    else:
        print("The specified worksheet is empty. No data to load.")

except Exception as e:
    print(f"An error occurred during Google Sheet loading: {e}")
    print("Please ensure you have authenticated successfully, the spreadsheet URL and worksheet GID are correct, and the service account has appropriate permissions.")

Authenticated successfully with Google Colab.
Google Sheet loaded successfully into DataFrame 'df'.
First 5 rows of the DataFrame (after improved alignment):


,source_label,email_date,sender,subject,gmail_message_id,thread_id,raw_body,raw_snippet,has_attachments,attachment_count,ingested_at,parse_status
0,20/05/2026,Alibaba <credit@notice.alibaba.com>,【Action Required】Your Trade Assurance Order No...,19e4562f7b098846,19e4562f7b098846,\n<https://m.alibaba.com?mt=mail&crm_mtn_trace...,\n<https://m.alibaba.com?mt=mail&crm_mtn_trace...,FALSE,21/05/2026,raw_ingested,None,None
1,13/05/2026,Alibaba <credit@notice.alibaba.com>,【Action Required】Your Trade Assurance Order No...,19e1ef8bab06e47f,19e1ef8bab06e47f,\n<https://m.alibaba.com?mt=mail&crm_mtn_trace...,\n<https://m.alibaba.com?mt=mail&crm_mtn_trace...,FALSE,21/05/2026,raw_ingested,None,None
2,16/04/2026,Alibaba <credit@notice.alibaba.com>,【Action Required】Your Trade Assurance Order No...,19d9432ae9a644eb,19d9432ae9a644eb,\n<https://m.alibaba.com?mt=mail&crm_mtn_trace...,\n<https://m.alibaba.com?mt=mail&crm_mtn_trace...,FALSE,21/05/2026,raw_ingested,None,None
3,14/04/2026,"""Alibaba.com"" <credit@notice.alibaba.com>",【Action Required】Your Trade Assurance order 29...,19d8ba45cbe3e605,19d8ba45cbe3e605,\nTrade Assurance\n【Action Required】Your Trade...,\nTrade Assurance\n【Action Required】Your Trade...,FALSE,21/05/2026,raw_ingested,None,None
4,14/04/2026,Alibaba <credit@notice.alibaba.com>,【Action Required】Your Trade Assurance Order No...,19d8ba41ee2d475d,19d8ba41ee2d475d,\n<https://m.alibaba.com?mt=mail&crm_mtn_trace...,\n<https://m.alibaba.com?mt=mail&crm_mtn_trace...,FALSE,21/05/2026,raw_ingested,None,None


In [24]:
import re

# Define the regex pattern to find the order number
# It looks for the specific phrase and then captures digits that follow it.
order_pattern = r'【Action Required】Your Trade Assurance Order No\. (\d+)'

# Apply the regex to the 'subject' column to extract the order numbers
# .str.extract() returns a DataFrame with one column for each captured group
order_numbers_df = df['subject'].str.extract(order_pattern)

# Rename the column for clarity
order_numbers_df.columns = ['order_number']

print("Extracted Order Numbers DataFrame:")
display(order_numbers_df.head())

Extracted Order Numbers DataFrame:


,order_number
0,NaN
1,NaN
2,NaN
3,NaN
4,NaN


In [28]:
import re

# Combine the original DataFrame with the extracted order numbers from subject
df_with_order_numbers = pd.concat([df, order_numbers_df], axis=1)
df_with_order_numbers = df_with_order_numbers.rename(columns={'order_number': 'subject_order_number'})

def extract_details_from_raw_body(raw_body_text):
    details = {
        'order_type': None,
        'order_number_raw_body': None,
        'order_date': None,
        'initial_payment_amount': None,
        'total_amount': None,
        'remaining_balance': None,
        'shipping_fee': None,
        'shipping_address': None,
        'shipping_method': None,
        'order_summary_items': None
    }

    if not isinstance(raw_body_text, str):
        return details

    # Order Type - This pattern is from the user's explicit request, not directly in the raw_body sample
    # This is unlikely to be found in the raw_body based on provided examples.
    order_type_match = re.search(r'Order Type:\s*(.*)', raw_body_text, re.DOTALL)
    if order_type_match:
        details['order_type'] = order_type_match.group(1).strip()

    # Order Number from raw_body (similar to subject but from a different context within raw_body)
    order_number_raw_body_match = re.search(r'Order No\.\s*(\d+)', raw_body_text, re.DOTALL)
    if order_number_raw_body_match:
        details['order_number_raw_body'] = order_number_raw_body_match.group(1)

    # Order Date - Adjusted to handle newlines between label and value
    order_date_match = re.search(r'Order date\s*([\d-]{10}\s[\d:]{8}\s[A-Z]{3})', raw_body_text, re.DOTALL)
    if order_date_match:
        details['order_date'] = order_date_match.group(1)

    # Initial Payment Amount
    initial_payment_match = re.search(r'Initial payment amount:\s*([A-Z]{3}\s[\d,\.]+)', raw_body_text, re.DOTALL)
    if initial_payment_match:
        details['initial_payment_amount'] = initial_payment_match.group(1)

    # Total Amount - Adjusted to handle newlines between label and value
    total_amount_match = re.search(r'Total\s*([A-Z]{3}\s[\d,\.]+)', raw_body_text, re.DOTALL)
    if total_amount_match:
        details['total_amount'] = total_amount_match.group(1)

    # Remaining Balance - Adjusted to handle newlines between label and value
    remaining_balance_match = re.search(r'Remaining balance:\s*([A-Z]{3}\s[\d,\.]+)', raw_body_text, re.DOTALL)
    if remaining_balance_match:
        details['remaining_balance'] = remaining_balance_match.group(1)

    # Shipping Fee - Adjusted to handle newlines between label and value
    shipping_fee_match = re.search(r'Shipping fee\s*(.*)', raw_body_text, re.DOTALL)
    if shipping_fee_match:
        details['shipping_fee'] = shipping_fee_match.group(1).strip()

    # Shipping Address - Adjusted to capture content between 'Shipping address' and a subsequent section/double newline
    shipping_address_match = re.search(r'Shipping address\s*([\s\S]*?)(?=\n{2,}|\nShipping method|\nOrder summary)', raw_body_text, re.DOTALL)
    if shipping_address_match:
        details['shipping_address'] = shipping_address_match.group(1).strip()

    # Shipping Method - Adjusted to handle newlines between label and value
    shipping_method_match = re.search(r'Shipping method\s*(.*)', raw_body_text, re.DOTALL)
    if shipping_method_match:
        details['shipping_method'] = shipping_method_match.group(1).strip()

    # Order Summary (items count) - Adjusted to handle newlines between label and value if any
    order_summary_items_match = re.search(r'Order summary \((\d+)\s*items\)', raw_body_text, re.DOTALL)
    if order_summary_items_match:
        details['order_summary_items'] = int(order_summary_items_match.group(1))

    return details

# Apply the extraction function to the 'raw_body' column
extracted_details_df = df_with_order_numbers['raw_body'].apply(extract_details_from_raw_body)

# Convert the series of dictionaries to a DataFrame
extracted_details_df = extracted_details_df.apply(pd.Series)

# Combine the original DataFrame with the new extracted details
df_final = pd.concat([df_with_order_numbers, extracted_details_df], axis=1)

print("DataFrame with all extracted details:")
display(df_final.head())

DataFrame with all extracted details:


,source_label,email_date,sender,subject,gmail_message_id,thread_id,raw_body,raw_snippet,has_attachments,attachment_count,...,order_type,order_number_raw_body,order_date,initial_payment_amount,total_amount,remaining_balance,shipping_fee,shipping_address,shipping_method,order_summary_items
0,20/05/2026,Alibaba <credit@notice.alibaba.com>,【Action Required】Your Trade Assurance Order No...,19e4562f7b098846,19e4562f7b098846,\n<https://m.alibaba.com?mt=mail&crm_mtn_trace...,\n<https://m.alibaba.com?mt=mail&crm_mtn_trace...,FALSE,21/05/2026,raw_ingested,...,None,302649438001028908,None,USD 1000.0000,None,None,None,None,None,None
1,13/05/2026,Alibaba <credit@notice.alibaba.com>,【Action Required】Your Trade Assurance Order No...,19e1ef8bab06e47f,19e1ef8bab06e47f,\n<https://m.alibaba.com?mt=mail&crm_mtn_trace...,\n<https://m.alibaba.com?mt=mail&crm_mtn_trace...,FALSE,21/05/2026,raw_ingested,...,None,300968051001028908,None,USD 420.0000,None,None,None,None,None,None
2,16/04/2026,Alibaba <credit@notice.alibaba.com>,【Action Required】Your Trade Assurance Order No...,19d9432ae9a644eb,19d9432ae9a644eb,\n<https://m.alibaba.com?mt=mail&crm_mtn_trace...,\n<https://m.alibaba.com?mt=mail&crm_mtn_trace...,FALSE,21/05/2026,raw_ingested,...,None,298896450001028908,None,USD 600.5200,None,None,None,None,None,None
3,14/04/2026,"""Alibaba.com"" <credit@notice.alibaba.com>",【Action Required】Your Trade Assurance order 29...,19d8ba45cbe3e605,19d8ba45cbe3e605,\nTrade Assurance\n【Action Required】Your Trade...,\nTrade Assurance\n【Action Required】Your Trade...,FALSE,21/05/2026,raw_ingested,...,None,None,None,None,None,None,None,None,None,None
4,14/04/2026,Alibaba <credit@notice.alibaba.com>,【Action Required】Your Trade Assurance Order No...,19d8ba41ee2d475d,19d8ba41ee2d475d,\n<https://m.alibaba.com?mt=mail&crm_mtn_trace...,\n<https://m.alibaba.com?mt=mail&crm_mtn_trace...,FALSE,21/05/2026,raw_ingested,...,None,297668291501028908,None,USD 118.0000,None,None,None,None,None,None
